In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nba_api.stats.endpoints import leaguedashteamstats
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.preprocessing import MinMaxScaler, Binarizer
from sklearn.metrics import confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


def fetch_team_data(seasons=None):
    """Fetch NBA team data from multiple seasons."""
    if seasons is None:
        seasons = [
            '2019-20',
            '2020-21',
            '2021-22',
            '2022-23',
            '2023-24',
            '2024-25'
        ]

    print(f"Fetching NBA team data for {len(seasons)} seasons...")

    all_dfs = []

    for season in seasons:
        print(f"  -> Fetching {season}")
        team_stats = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced'
        )
        df = team_stats.get_data_frames()[0]
        df['SEASON'] = season  # 👈 track season
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)

    print(f"Total rows collected: {len(combined_df)}")

    print('raw data')
    print(combined_df)
    return combined_df


def prepare_data(df):
    """Prepare features and create target variable."""
    features = [
        'OFF_RATING',
        'DEF_RATING',
        'NET_RATING',
        'PACE',
        'TS_PCT',
        'EFG_PCT',
        'AST_RATIO',
        'REB_PCT',
        'TM_TOV_PCT',
        'PIE'
    ]

    df_clean = df[['TEAM_NAME'] + features + ['W_PCT']].dropna()
    df_clean['WIN_TIER'] = pd.qcut(
        df_clean['W_PCT'],
        3,
        labels=['Low', 'Mid', 'High'],
        duplicates='drop'
    )

    X = df_clean[features].copy()
    y = df_clean['WIN_TIER']
    team_names = df_clean['TEAM_NAME']

    print(f"Data shape: {X.shape}")
    print(f"Target distribution:\n{y.value_counts().sort_index()}")

    print('train test split')
    print(X)
    print(y)

    return X, y, team_names


def plot_labeled_confusion_matrix(y_true, y_pred, class_names, title, filename):
    """Plot and save a labeled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=class_names)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved: {filename}")


def run_multinomial_nb(X_train, X_test, y_train, y_test, class_names):
    """Run Multinomial Naive Bayes (requires non-negative features)."""
    print("\n--- Multinomial NB ---")

    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf = MultinomialNB()
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)

    print(classification_report(y_test, y_pred, target_names=class_names))

    plot_labeled_confusion_matrix(
        y_test, y_pred, class_names,
        'Multinomial NB - Confusion Matrix',
        'outputs/nb_multinomial_confusion.png'
    )

    return clf


def run_gaussian_nb(X_train, X_test, y_train, y_test, class_names):
    """Run Gaussian Naive Bayes (assumes normal distribution)."""
    print("\n--- Gaussian NB ---")
    clf = GaussianNB()
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    print(classification_report(y_test, y_pred, target_names=class_names))

    plot_labeled_confusion_matrix(
        y_test, y_pred, class_names,
        'Gaussian NB - Confusion Matrix',
        'outputs/nb_gaussian_confusion.png'
    )

    return clf


def run_bernoulli_nb(X_train, X_test, y_train, y_test, class_names):
    """Run Bernoulli Naive Bayes (requires binary features)."""
    print("\n--- Bernoulli NB ---")

    binarizer = Binarizer(threshold=X_train.median().median())
    X_train_binary = binarizer.fit_transform(X_train)
    X_test_binary = binarizer.transform(X_test)

    clf = BernoulliNB()
    clf.fit(X_train_binary, y_train)

    y_pred = clf.predict(X_test_binary)

    print(classification_report(y_test, y_pred, target_names=class_names))

    plot_labeled_confusion_matrix(
        y_test, y_pred, class_names,
        'Bernoulli NB - Confusion Matrix',
        'outputs/nb_bernoulli_confusion.png'
    )

    return clf


def main():
    df = fetch_team_data()
    X, y, team_names = prepare_data(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    class_names = y.unique()

    run_multinomial_nb(X_train, X_test, y_train, y_test, class_names)
    run_gaussian_nb(X_train, X_test, y_train, y_test, class_names)
    run_bernoulli_nb(X_train, X_test, y_train, y_test, class_names)

    print("\n=== Summary ===")
    print("All confusion matrices saved to /outputs folder:")
    print("  - nb_multinomial_confusion.png (with MinMaxScaler)")
    print("  - nb_gaussian_confusion.png (no transformation)")
    print("  - nb_bernoulli_confusion.png (with binarization)")


if __name__ == "__main__":
    main()


Fetching NBA team data for 6 seasons...
  -> Fetching 2019-20
  -> Fetching 2020-21
  -> Fetching 2021-22
  -> Fetching 2022-23
  -> Fetching 2023-24
  -> Fetching 2024-25
Total rows collected: 180
raw data
        TEAM_ID           TEAM_NAME  GP   W   L  W_PCT     MIN  E_OFF_RATING  \
0    1610612737       Atlanta Hawks  67  20  47  0.299  3256.0         104.3   
1    1610612738      Boston Celtics  72  48  24  0.667  3486.0         110.4   
2    1610612751       Brooklyn Nets  72  35  37  0.486  3496.0         106.0   
3    1610612766   Charlotte Hornets  65  23  42  0.354  3150.0         103.9   
4    1610612741       Chicago Bulls  65  22  43  0.338  3135.0         104.1   
..          ...                 ...  ..  ..  ..    ...     ...           ...   
175  1610612758    Sacramento Kings  82  40  42  0.488  3976.0         113.8   
176  1610612759   San Antonio Spurs  82  34  48  0.415  3946.0         111.7   
177  1610612761     Toronto Raptors  82  30  52  0.366  3961.0         10